# Machine Learning with FiberNet

This tutorial demonstrates machine learning workflows:

1. Feature extraction from fiber networks
2. Training property prediction models
3. GNN feature extraction for graph neural networks
4. Building datasets for ML

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fibernet import gen
from fibernet.ml import FeatureExtractor
from fibernet.analysis import MorphologyAnalyzer

## 1. Generate a Dataset of Networks

In [ ]:
# Generate 100 networks with varying density
networks = []
labels = []

for i in range(100):
    num_fibers = np.random.randint(50, 200)
    fiber_length = np.random.uniform(8, 15)
    net = gen.random_straight_2d(
        num_fibers=num_fibers,
        fiber_length=fiber_length,
        box_size=(50, 50),
        seed=i
    )
    networks.append(net)
    # Use connectivity as a target property
    analyzer = MorphologyAnalyzer(net)
    labels.append(analyzer.mean_connectivity())

print(f'Dataset: {len(networks)} networks')

## 2. Extract Features

In [ ]:
extractor = FeatureExtractor()

X = []
for net in networks:
    features = extractor.extract_features(net)
    X.append(features)

X = np.array(X)
y = np.array(labels)

print(f'Feature matrix shape: {X.shape}')
print(f'Labels shape: {y.shape}')

## 3. Train a Prediction Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'RMSE: {rmse:.4f}')
print(f'R2 score: {r2:.4f}')

## 4. GNN Feature Extraction

In [ ]:
from fibernet.ml import GNNFeatureExtractor

gnn_extractor = GNNFeatureExtractor(
    node_features=['position', 'degree'],
    edge_features=['length', 'angle']
)

graph_data = gnn_extractor.extract_graph(networks[0])
print(f'Node features: {graph_data["node_features"].shape}')
print(f'Edge index: {graph_data["edge_index"].shape}')
print(f'Num edges: {graph_data["num_edges"]}')

## 5. Visualize Structure-Property Relationships

In [ ]:
densities = []
orders = []

for net in networks:
    a = MorphologyAnalyzer(net)
    densities.append(a.fiber_density())
    orders.append(a.nematic_order_parameter())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(densities, y, alpha=0.6)
axes[0].set_xlabel('Fiber Density')
axes[0].set_ylabel('Connectivity')
axes[1].scatter(orders, y, alpha=0.6, color='orange')
axes[1].set_xlabel('Nematic Order')
axes[1].set_ylabel('Connectivity')
plt.tight_layout()
plt.show()

## Summary

In this tutorial you learned:
- Feature extraction from fiber networks
- Training ML models for property prediction
- GNN feature extraction for graph neural networks
- Visualizing structure-property relationships